In [0]:
display(dbutils.fs.ls("/Volumes/cert/bronze/raw/dados/dados_databricks_pratica"))

In [0]:
display(dbutils.fs.ls("/Volumes/cert/bronze/raw/"))

In [0]:
display(dbutils.fs.ls("/Volumes/cert/bronze/raw/dados/"))

In [0]:
display(dbutils.fs.ls("/Volumes/cert/bronze/raw/dados_databricks_pratica/"))

In [0]:
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/cert/bronze/raw/_schema/vendas")
    .option("header", "true")
    .load("/Volumes/cert/bronze/raw/dados_databricks_pratica/")
)

(df_stream.writeStream
    .option("checkpointLocation", "/Volumes/cert/bronze/raw/_checkpoint/vendas")
    .trigger(availableNow=True)
    .toTable("cert.bronze.tabela_autoloader")
)

In [0]:
%sql
SELECT * FROM cert.bronze.tabela_autoloader;

In [0]:
dbutils.fs.rm("/Volumes/cert/bronze/raw/_schema/vendas", recurse=True)
dbutils.fs.rm("/Volumes/cert/bronze/raw/_checkpoint/vendas", recurse=True)

In [0]:
%sql
DROP TABLE IF EXISTS cert.bronze.tabela_autoloader;

In [0]:
dbutils.fs.mkdirs("/Volumes/cert/bronze/raw/vendas_raw/")
dbutils.fs.cp(
    "/Volumes/cert/bronze/raw/dados_databricks_pratica/vendas.csv",
    "/Volumes/cert/bronze/raw/vendas_raw/vendas.csv"
)

In [0]:
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/cert/bronze/raw/_schema/vendas")
    .option("header", "true")
    .load("/Volumes/cert/bronze/raw/vendas_raw/")
)

(df_stream.writeStream
    .option("checkpointLocation", "/Volumes/cert/bronze/raw/_checkpoint/vendas")
    .trigger(availableNow=True)
    .toTable("cert.bronze.vendas_bronze")
)

In [0]:
%sql
SELECT * FROM cert.bronze.vendas_bronze

In [0]:
dbutils.fs.cp(
    "/Volumes/cert/bronze/raw/dados_databricks_pratica/vendas_com_problemas.csv",
    "/Volumes/cert/bronze/raw/vendas_raw/vendas_com_problemas.csv"
)

In [0]:
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/cert/bronze/raw/_schema/vendas")
    .option("header", "true")
    .load("/Volumes/cert/bronze/raw/vendas_raw/")
)

(df_stream.writeStream
    .option("checkpointLocation", "/Volumes/cert/bronze/raw/_checkpoint/vendas")
    .trigger(availableNow=True)
    .toTable("cert.bronze.vendas_bronze")
)

In [0]:
%sql
SELECT * FROM cert.bronze.vendas_bronze

In [0]:
%sql
SELECT * FROM cert.bronze.vendas_bronze
WHERE venda_id IN ('V1002','V1003','V1005','V1006','V1007','V1008')

In [0]:
%sql
SELECT venda_id, quantidade,
       TRY_CAST(quantidade AS INT) AS quantidade_valida
FROM cert.bronze.vendas_bronze
ORDER BY venda_id